# Práctica 4B: Búsqueda por similaridad

**Realizado por: Eduardo Alanís García**


## Instrucciones

En esta práctica se implementará la búsqueda por similaridad de texto. El conjunto de datos son títulos de noticias. Se usarán modelos de lenguaje para obtener los encajes del texto y formar la base de datos. Luego se harán consultas para buscar los elementos más cercanos y reportarlos.

1. Lea el conjunto de datos: [archivo_emojis_Elfinanciero.csv](https://aulavirtual.infotec.mx/pluginfile.php/159734/mod_assign/intro/archivo_emojis_Elfinanciero.csv?time=1769718403606) Se trabajará sólo con la columna `title`.
2. Debe usar modelos de lenguaje para convertir los títulos en vectores y formar la base de datos . Los modelos a usar son:
   - Beto: “dccuchile/bert-base-spanish-wwm-cased”
   - Robertuito: “pysentimiento/robertuito-base-cased”
3. Usa las siguientes consultas para buscar los 5 elementos más cercanos en . No se necesita ningún preprocesado ya que de esto se encarga el tokenizador.

```json
["Muerte de migrantes en un incendio en Ciudad Juárez",
"¿Qué tan sano es comer tacos de cabeza?",
"La crisis humanitaria de Estados Unidos",
"Lo que hay del otro laredo",
"¿Existe un acuerdo entre la banca tradicional y los nuevos jugadores?",
"Maribel Guardia afirma: Ya no le tengo miedo",
"Graban pelea en Xochimilco (Video)"]
```

5. Por cada consulta, mostrar los cinco títulos de noticias más parecidos en  usnado la métrica coseno e incluir el score de similaridad.
6. Contesta la evaluación de esta práctica en el Aula Virtual.

Puede leer los modelos con el código:

```python
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path)
model = model.to(device)
```

Para obtener los encajes puede usar la siguiente función:

```python
def predict(text, bs=128):
    output = []
    for i in range(0, len(text), bs):
        if i//bs%100==99:
            print(i, "/", len(text))
        tokens = tokenizer(text[i: i+bs], return_tensors="pt", padding='max_length', max_length=50, truncation=True)
        t = {"input_ids": tokens['input_ids'].to(device), "attention_mask": tokens['attention_mask'].to(device)}

        with torch.no_grad():
            pred = model(**t).last_hidden_state[:,0].cpu()
        output.append(pred)
    output = torch.cat(output, dim=0)
    return output
```

### Bibliotecas y modelos



In [1]:
%pip install -q torch transformers


In [2]:
import os
import gc
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel


### Carga de datos




In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
direccion = "/content/drive/MyDrive/MCDI-INFOTEC/1. Primer semestre/Manejo y procesamiento de la informacion/Unidad 4/4B. Práctica Búsqueda por similaridad/"
ruta_archivo = os.path.join(direccion, "archivo_emojis_Elfinanciero.csv")

datos = pd.read_csv(ruta_archivo, usecols=["title"])
datos = datos.dropna(subset=["title"])
titulos = datos["title"].astype(str).tolist()

print("Títulos disponibles:", len(titulos))
datos.head()


Títulos disponibles: 27831


,title
0,AMLO: otra duda sobre su cardenismo
1,Esto sabemos de la muerte de 39 migrantes en u...
2,Ricardo Monreal instala primer comité de proye...
3,"Israel, en tiempo extra"
4,Alfonso Herrera y su miedo tras tragedia de RB...


### Consultas



In [5]:
consultas = [
    "Muerte de migrantes en un incendio en Ciudad Juárez",
    "¿Qué tan sano es comer tacos de cabeza?",
    "La crisis humanitaria de Estados Unidos",
    "Lo que hay del otro laredo",
    "¿Existe un acuerdo entre la banca tradicional y los nuevos jugadores?",
    "Maribel Guardia afirma: Ya no le tengo miedo",
    "Graban pelea en Xochimilco (Video)"
]



### Cargar los modelos y obtener los encajes

El encaje del texto se toma del primer token de `last_hidden_state`.


In [6]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)


def predict(text, bs=128):
  output = []

  for i in range(0, len(text), bs):
    if i // bs % 100 == 99:
      print(i, "/", len(text))

    tokens = tokenizer(
        text[i: i + bs],
        return_tensors="pt",
        padding="max_length",
        max_length=50,
        truncation=True
    )

    t = {
        "input_ids": tokens["input_ids"].to(device),
        "attention_mask": tokens["attention_mask"].to(device)
    }

    with torch.no_grad():
      pred = model(**t).last_hidden_state[:, 0].cpu()

    output.append(pred)

  output = torch.cat(output, dim=0)
  return output


def obtener_resultados(embeddings_titulos, embeddings_consultas, titulos, consultas, nombre_modelo):

  embeddings_titulos = torch.nn.functional.normalize(embeddings_titulos, p=2, dim=1)
  embeddings_consultas = torch.nn.functional.normalize(embeddings_consultas, p=2, dim=1)
  similitudes = embeddings_consultas @ embeddings_titulos.T  # Sabemos que la norma es 1 de cada vector

  resultados = []

  for i, consulta in enumerate(consultas):
    indices = torch.topk(similitudes[i], k=5).indices.tolist()

    for posicion, indice in enumerate(indices, start=1):
      resultados.append({
          "modelo": nombre_modelo,
          "consulta": consulta,
          "posición": posicion,
          "título más parecido": titulos[indice],
          "similitud coseno": similitudes[i, indice].item()
      })

  return pd.DataFrame(resultados)


Dispositivo: cuda:0


### Beto

Se obtienen los encajes de los títulos y de las consultas con `dccuchile/bert-base-spanish-wwm-cased`. Para cada consulta, `torch.topk` recupera los cinco scores más altos.


In [8]:
model_path = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path)
model = model.to(device)
model.eval()

encajes_titulos_beto = predict(titulos, bs=128)
encajes_consultas_beto = predict(consultas, bs=128)

resultados_beto = obtener_resultados(
    encajes_titulos_beto,
    encajes_consultas_beto,
    titulos,
    consultas,
    "Beto"
)

resultados_beto


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


12672 / 27831
25472 / 27831


,modelo,consulta,posición,título más parecido,similitud coseno
0,Beto,Muerte de migrantes en un incendio en Ciudad J...,1,Nueva tragedia migrante: Muere un haitiano en ...,0.918477
1,Beto,Muerte de migrantes en un incendio en Ciudad J...,2,El brutal asesinato de una bebé de dos años du...,0.911774
2,Beto,Muerte de migrantes en un incendio en Ciudad J...,3,Violencia en Nuevo León: Ataque en centro noct...,0.908202
3,Beto,Muerte de migrantes en un incendio en Ciudad J...,4,Violencia en Nuevo León: Ataque en centro noct...,0.908202
4,Beto,Muerte de migrantes en un incendio en Ciudad J...,5,Masacre en Nuevo León: Asesinan a seis persona...,0.906143
5,Beto,¿Qué tan sano es comer tacos de cabeza?,1,¿Qué tan sanos son los chiles en nogada?,0.970252
6,Beto,¿Qué tan sano es comer tacos de cabeza?,2,¿Qué tan saludable es la barbacoa?,0.966751
7,Beto,¿Qué tan sano es comer tacos de cabeza?,3,¿Qué tanto daño hace comer papas fritas?,0.965992
8,Beto,¿Qué tan sano es comer tacos de cabeza?,4,¿Qué tanto daño hace comer tacos de tripa?,0.964099
9,Beto,¿Qué tan sano es comer tacos de cabeza?,5,¿Qué tanto daño hacen las alitas de pollo?,0.962858


### Robertuito

Se repite el mismo procedimiento con `pysentimiento/robertuito-base-cased`


In [9]:
model_path = "pysentimiento/robertuito-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path)
model = model.to(device)
model.eval()

encajes_titulos_robertuito = predict(titulos, bs=128)
encajes_consultas_robertuito = predict(consultas, bs=128)

resultados_robertuito = obtener_resultados(
    encajes_titulos_robertuito,
    encajes_consultas_robertuito,
    titulos,
    consultas,
    "Robertuito"
)

resultados_robertuito


config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/319 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/828k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  435MB            

...

[transformers] RobertaModel LOAD REPORT from: pysentimiento/robertuito-base-cased
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors: reconstructing file:   0%|          |  0.00B /  435MB            

model.safetensors: downloading bytes:           |  0.00B            

12672 / 27831
25472 / 27831


,modelo,consulta,posición,título más parecido,similitud coseno
0,Robertuito,Muerte de migrantes en un incendio en Ciudad J...,1,Mueren siete personas indígenas desplazadas en...,0.907657
1,Robertuito,Muerte de migrantes en un incendio en Ciudad J...,2,Incendio en Ciudad Juárez: Vinculan a proceso ...,0.904213
2,Robertuito,Muerte de migrantes en un incendio en Ciudad J...,3,"Ataque con drones y bombas en Tierra Caliente,...",0.902052
3,Robertuito,Muerte de migrantes en un incendio en Ciudad J...,4,Muerte de migrantes en Ciudad Juárez: Así fue ...,0.898988
4,Robertuito,Muerte de migrantes en un incendio en Ciudad J...,5,Incendio en Ciudad Juárez: Revelan identidad d...,0.898305
5,Robertuito,¿Qué tan sano es comer tacos de cabeza?,1,¿Qué tan sano es comer tacos de pastor?,0.974449
6,Robertuito,¿Qué tan sano es comer tacos de cabeza?,2,¿Qué tanto daño hace comer tacos de tripa?,0.923205
7,Robertuito,¿Qué tan sano es comer tacos de cabeza?,3,¿Qué tan peligroso es comer ceviche?,0.914443
8,Robertuito,¿Qué tan sano es comer tacos de cabeza?,4,¿Qué tan peligroso es comer ceviche?,0.914443
9,Robertuito,¿Qué tan sano es comer tacos de cabeza?,5,¿Qué tan sanos son los chiles en nogada?,0.911032


### Contesta la evaluación de esta práctica en el Aula Virtual.

1. Usando el tokenizador del modelo dccuchile/bert-base-spanish-wwm-cased, al tokenizar el texto "Buenos días Aguascalientes", obtenemos los tokens

In [10]:
model_path = "dccuchile/bert-base-spanish-wwm-cased"

In [11]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path)
model = model.to(device)

cuda:0


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
text = ["Buenos días Aguascalientes"]
tokens = tokenizer(text, return_tensors="pt", padding='max_length', max_length=20, truncation=True)


In [14]:
tokens

{'input_ids': tensor([[    4,  4189,  2045, 23107, 18721,  1314,     5,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}

2. Usando el tokenizador del modelo dccuchile/bert-base-spanish-wwm-cased, decodificar los tokens [4, 1716, 1013, 2073, 16718, 1011, 1009, 5]

In [16]:
tokenizer.decode([4, 1716, 1013, 2073, 16718, 1011, 1009, 5])

'[CLS] Vamos a comer tacos. [SEP]'

3. Usando el tokenizador del modelo pysentimiento/robertuito-base-cased, al tokenizar el texto "Buenos días Aguascalientes", obtenemos los tokens

In [25]:
model_path = "pysentimiento/robertuito-base-cased"

In [26]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModel.from_pretrained(model_path)
model = model.to(device)

cuda:0


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: pysentimiento/robertuito-base-cased
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [27]:
text = ["Buenos días Aguascalientes"]
tokens = tokenizer(text, return_tensors="pt", padding='max_length', max_length=20, truncation=True)

In [30]:
tokens

{'input_ids': tensor([[    0,  2625,  1115, 27194,  1316,  1140,     2,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}

4. Usando el tokenizador del modelo pysentimiento/robertuito-base-cased, al decodificar los tokens [0, 2493, 417, 1707, 13105, 2] obtenemos:

In [31]:
tokenizer.decode([0, 2493, 417, 1707, 13105, 2])

'<s> Vamos a comer tacos</s>'